# 02 — Full-data DEV train: giữ nguyên One-Hot category, dùng sparse float64

**Mục tiêu:** Dùng toàn bộ TRAIN (302.471 hàng) và VAL (64.826 hàng), giữ **đầy đủ category** được phép trong contract, giữ `CustomRandomForestRegressor` và tham số đã định, chỉ đổi cách **lưu ma trận từ dense sang sparse** để tiết kiệm RAM. Không dùng `max_categories`, `min_frequency` hay gộp nhóm category.

**Trước khi train full:** notebook tự kiểm tra rằng One-Hot dense và sparse mã hóa **cùng giá trị/đúng thứ tự cột** và chạy thử cùng Random Forest trên một tập nhỏ; nếu dự đoán không tương đương trong ngưỡng nghiêm ngặt, notebook sẽ **dừng**.

**Giới hạn:** Đây vẫn là lần DEV vì `characteristics` và `body_type_name` đang thiếu trong Parquet; hai cột này không thể được khôi phục bằng cách đổi sparse. Một số cột ngoài contract (`car_age`, `years_into_gen`, `gen_in_production`) vẫn không nằm trong mô hình, theo notebook trước. Không đụng tới TEST. Không thể bảo đảm kết quả full-fit giống từng bit giữa dense và sparse trước khi thử thực tế trên toàn bộ dữ liệu.


## 1. Cấu hình và import từ source gốc của nhóm

In [ ]:
import json
import os
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from scipy import sparse

# Dùng đúng source trong repo đã clone; hỗ trợ chạy local từ thư mục gốc hoặc notebooks/.
REPO = Path('/kaggle/working/MachineLearningRandomForest')
if not (REPO / 'src' / 'mlrf').is_dir():
    REPO = Path.cwd().resolve()
    if REPO.name == 'notebooks':
        REPO = REPO.parent
if not (REPO / 'src' / 'mlrf').is_dir():
    raise FileNotFoundError('Không tìm thấy repo có src/mlrf. Hãy chạy notebook từ root hoặc notebooks/ của repo.')

os.chdir(REPO)  # load_project_config() / load_ds04_contract() có thể dùng đường dẫn tương đối.
sys.path.insert(0, str(REPO / 'src'))

from mlrf.config import load_project_config
from mlrf.data.contract import load_ds04_contract
from mlrf.evaluation import regression_metrics
from mlrf.ops import environment_info, save_run
from mlrf.testing import assert_target_absent
from mlrf.training import build_model

CONF = load_project_config()
CONTRACT = load_ds04_contract()
SEED = int(CONF['split']['random_seed'])
TARGET = CONF['dataset']['target']

DATA_DIR = Path(os.environ.get('MLRF_DATA_DIR', '/kaggle/input/datasets/ncd426/machinelearning'))
MAX_TRAIN_ROWS = None         # Full train 302,471 dòng; không đổi sang dùng tập TEST.
MAX_VAL_ROWS = None          # Giữ None để đánh giá trên toàn bộ val.parquet.
MIN_AVAILABLE_GIB = 8
VAL_PRED_BATCH = 2_048       # Dự đoán VAL thành nhiều batch để tránh one-hot chiếm quá nhiều RAM.
RUN_ID = 'real_dev_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
MODEL_PARAMS = {
    'n_estimators': 12,
    'max_depth': 7,
    'max_features': 0.7,
    'bootstrap': True,
    'oob_score': True,
    'random_state': SEED,
}

print('Repo:', REPO)
print('Data:', DATA_DIR)
print('Target:', TARGET, '| Seed:', SEED)
print('Python/libraries:', environment_info())
print('Run ID:', RUN_ID)
print('DEV run: not final, dataset approval not claimed.')


## 2. Đọc đúng các split đã có, không chia lại

In [ ]:
required = ['train.parquet', 'val.parquet', 'test.parquet']
for name in required:
    path = DATA_DIR / name
    if not path.is_file():
        raise FileNotFoundError(f'Thiếu {path}. Hãy gắn Kaggle Dataset hoặc cấu hình MLRF_DATA_DIR.')

train_df = pd.read_parquet(DATA_DIR / 'train.parquet')
val_df = pd.read_parquet(DATA_DIR / 'val.parquet')

print('Train shape:', train_df.shape)
print('Val shape:', val_df.shape)
print('Test file:', DATA_DIR / 'test.parquet', '(chưa đọc nội dung test)')

# Hiện các báo cáo đi kèm để người phụ trách xác nhận trước khi tuyên bố kết quả cuối.
for name in ['cleaning_report.json', 'preprocessing_check.json', 'split_report.json']:
    p = DATA_DIR / name
    if p.exists():
        try:
            report = json.loads(p.read_text(encoding='utf-8'))
            keys = sorted(report.keys()) if isinstance(report, dict) else []
            print(f'{name}: tồn tại; keys = {keys[:30]}')
        except (ValueError, UnicodeError) as exc:
            print(f'{name}: không đọc được JSON: {exc}')

## 3. Kiểm tra schema, tạo X/y từ contract của dự án

In [ ]:
# Data contract vẫn là nguồn tham chiếu; không sửa dữ liệu, không thêm cột giả.
contract_numeric = [c for c in CONTRACT['numeric_columns'] if c != TARGET]
contract_categorical = [c for c in CONTRACT['categorical_columns'] if c != TARGET]
expected_features = contract_numeric + contract_categorical

if not expected_features:
    raise ValueError('Data contract không có feature để train.')
if len(expected_features) != len(set(expected_features)):
    raise ValueError('Data contract có feature bị trùng giữa numeric/categorical.')

# DEV waiver: chỉ hai cột này đang được báo thiếu trong dữ liệu thật.
# TODO: TV1/TV2 xác nhận schema chính thức trước khi xóa waiver.
DEV_MISSING_FEATURE_ALLOWLIST = {'characteristics', 'body_type_name'}
missing_by_split = {
    label: sorted(set(expected_features) - set(df.columns))
    for label, df in [('train', train_df), ('val', val_df)]
}
print('Data contract features:', len(expected_features))
print('Train columns:', train_df.columns.tolist())
print('Validation columns:', val_df.columns.tolist())
print('Missing features by split:', missing_by_split)

missing_train = set(missing_by_split['train'])
missing_val = set(missing_by_split['val'])
if missing_train != missing_val:
    raise ValueError('Train/val thiếu các feature KHÁC nhau; phải đối chiếu pipeline chia dữ liệu trước khi train.')
unexpected_missing = missing_train - DEV_MISSING_FEATURE_ALLOWLIST
if unexpected_missing:
    raise ValueError(f'Thiếu cột ngoài DEV allowlist: {sorted(unexpected_missing)}. Không tự bỏ cột.')

missing_contract_features = sorted(missing_train)
if missing_contract_features:
    print('WARNING DEV: Bỏ qua tạm thời feature thiếu trong Parquet:', missing_contract_features)
    print('WARNING DEV: KHÔNG được dùng metrics này làm kết quả cuối trước khi TV1/TV2 nghiệm thu.')

numeric = [c for c in contract_numeric if c not in missing_train]
categorical = [c for c in contract_categorical if c not in missing_train]
features = numeric + categorical
if not features:
    raise ValueError('Không còn feature để train sau khi đối chiếu schema.')

for label, df in [('train', train_df), ('val', val_df)]:
    if TARGET not in df.columns:
        raise ValueError(f'{label}.parquet thiếu cột mục tiêu {TARGET!r}.')
    if not pd.api.types.is_numeric_dtype(df[TARGET]):
        raise TypeError(f'{label}.{TARGET} chưa có kiểu số. Cần xử lý ở pipeline dữ liệu trước.')
    if df[TARGET].isna().any() or not np.isfinite(df[TARGET].to_numpy(dtype=float)).all():
        raise ValueError(f'{label}.{TARGET} có null/inf. Cần xử lý dữ liệu trước khi train.')
    wrong_numeric = [c for c in numeric if not pd.api.types.is_numeric_dtype(df[c])]
    if wrong_numeric:
        raise TypeError(f'{label}: cột numeric sai kiểu: {wrong_numeric}')

# Lấy mẫu chỉ TRONG tập train; không gộp hay chia lại các tập.
train_use = (train_df.sample(n=MAX_TRAIN_ROWS, random_state=SEED)
             if MAX_TRAIN_ROWS is not None and len(train_df) > MAX_TRAIN_ROWS
             else train_df)
val_use = (val_df.sample(n=MAX_VAL_ROWS, random_state=SEED)
           if MAX_VAL_ROWS is not None and len(val_df) > MAX_VAL_ROWS
           else val_df)

def make_x(df):
    x = df[features].copy()  # Không đưa cột target hay những cột ngoài contract vào model.
    for col in categorical:
        # Chuẩn hóa kiểu để SimpleImputer/OneHotEncoder không gặp hỗn hợp string + pd.NA.
        x[col] = x[col].astype('string').fillna('__MISSING__').astype(object)
    return x

X_train = make_x(train_use)
y_train = train_use[TARGET].copy()
X_val = make_x(val_use)
y_val = val_use[TARGET].copy()

assert_target_absent(X_train, target=TARGET)
assert_target_absent(X_val, target=TARGET)
print('Train used:', len(X_train), 'of', len(train_df))
print('Val used:', len(X_val), 'of', len(val_df))
print('Numeric:', numeric)
print('Categorical:', categorical)
print('Extra columns excluded:', sorted(set(train_df.columns) - set(features) - {TARGET}))

## 4. Bộ mã hóa One-Hot đầy đủ, lưu sparse float64

Ở đây **không** giới hạn số lượng category, không đổi tham số Random Forest và không sửa Parquet. `ColumnTransformer(sparse_threshold=1.0)` giữ kết quả sparse thay vì chuyển thành dense. Vì chỉ có khoảng 18 giá trị khác 0 trên một dòng, cách biểu diễn này có thể giảm RAM rất nhiều.

Notebook sẽ chạy một phép thử trên **1.000 dòng TRAIN và 128 dòng VAL**, kiểm tra mã hóa và so sánh dự đoán dense/sparse với cùng seed trước khi full train. Đây chỉ là kiểm tra tương đương trên mẫu, không phải chứng minh hai đường chạy hoàn toàn giống nhau trong mọi trường hợp.


In [ ]:
import psutil
from sklearn.base import clone

cardinality = {c: int(X_train[c].nunique(dropna=False)) for c in categorical}
feature_dim_estimate = len(numeric) + sum(cardinality.values())
print('All category cardinality (no grouping):', cardinality)
print('Estimated FULL one-hot feature count:', feature_dim_estimate)

# Upper bound for sparse nnz: each numeric value + 1 active one-hot per categorical column.
# Sparse CSR memory for float64 data + 32-bit column indices + row pointers.
max_nnz = len(X_train) * (len(numeric) + len(categorical))
upper_csr_mib = (max_nnz * (8 + 4) + (len(X_train) + 1) * 4) / 1024**2
print(f'Estimated sparse CSR TRAIN storage upper bound (input only): {upper_csr_mib:,.1f} MiB')
print('WARNING: This does NOT estimate all memory used by tree building, copies, pandas, and Python.')

def available_memory_mib():
    available = psutil.virtual_memory().available / 1024**2
    for base in [Path('/sys/fs/cgroup'), Path('/sys/fs/cgroup/memory')]:
        cap, used = base / 'memory.max', base / 'memory.current'
        if cap.exists() and used.exists():
            limit_text = cap.read_text().strip()
            if limit_text.isdigit():
                available = min(available, max(0, int(limit_text) - int(used.read_text().strip())) / 1024**2)
        cap, used = base / 'memory.limit_in_bytes', base / 'memory.usage_in_bytes'
        if cap.exists() and used.exists():
            limit_bytes = int(cap.read_text().strip())
            if limit_bytes < 1 << 60:
                available = min(available, max(0, limit_bytes - int(used.read_text().strip())) / 1024**2)
    return available

available_mb = available_memory_mib()
print(f'RAM available (cgroup-aware estimate): {available_mb / 1024:,.2f} GiB')
if available_mb < MIN_AVAILABLE_GIB * 1024:
    raise MemoryError(f'Available RAM < {MIN_AVAILABLE_GIB} GiB. Do not start full fit.')

# DENSE/Sparse preprocessing differ ONLY by matrix representation.
# Dtype float64, handle_unknown=ignore, no category limit.
def make_preprocessor(sparse_output=True):
    return ColumnTransformer([
        ('num', Pipeline([
            ('fill', SimpleImputer(strategy='median')),
        ]), numeric),
        ('cat', Pipeline([
            ('fill', SimpleImputer(strategy='most_frequent')),
            ('encode', OneHotEncoder(
                handle_unknown='ignore',
                sparse_output=sparse_output,
                dtype=np.float64,
            )),
        ]), categorical),
    ], sparse_threshold=1.0 if sparse_output else 0.0)

# Check 1: identical input features and dense/sparse encoding on a fixed sample.
PROBE_TRAIN_ROWS = min(1_000, len(X_train))
PROBE_VAL_ROWS = min(128, len(X_val))
X_probe, y_probe = X_train.iloc[:PROBE_TRAIN_ROWS], y_train.iloc[:PROBE_TRAIN_ROWS]
X_probe_val = X_val.iloc[:PROBE_VAL_ROWS]

pre_dense = make_preprocessor(sparse_output=False)
pre_sparse = make_preprocessor(sparse_output=True)
D = pre_dense.fit_transform(X_probe)
S = pre_sparse.fit_transform(X_probe)
if not sparse.issparse(S):
    raise RuntimeError('Sparse preprocessing produced a dense matrix. STOP.')
if D.shape != S.shape or not np.array_equal(D, S.toarray()):
    raise RuntimeError('Dense and sparse preprocessing differ on probe sample. STOP.')
assert D.dtype == np.float64 and S.dtype == np.float64
print('PROBE preprocessing identical: float64, shape:', D.shape, '| sparse nnz:', S.nnz)

# Check 2: inner RandomForest and DecisionTree must fit/predict sparse data.
# Explicitly compare predictions using the SAME algorithm and random seed.
model_dense_probe = Pipeline([
    ('preprocess', make_preprocessor(sparse_output=False)),
    ('regressor', build_model('custom_random_forest', MODEL_PARAMS)),
])
model_sparse_probe = Pipeline([
    ('preprocess', make_preprocessor(sparse_output=True)),
    ('regressor', build_model('custom_random_forest', MODEL_PARAMS)),
])
try:
    model_dense_probe.fit(X_probe, y_probe)
    model_sparse_probe.fit(X_probe, y_probe)
    pred_dense_probe = model_dense_probe.predict(X_probe_val)
    pred_sparse_probe = model_sparse_probe.predict(X_probe_val)
except Exception as exc:
    raise RuntimeError(
        'Custom Random Forest or inner tree may not support sparse end-to-end; '
        'do not run full train until src is checked.'
    ) from exc

max_diff = float(np.max(np.abs(pred_dense_probe - pred_sparse_probe)))
print('PROBE dense/sparse max absolute prediction difference:', max_diff)
# Strict absolute threshold on price predictions; this is NOT a proof for the full data.
if not np.allclose(pred_dense_probe, pred_sparse_probe, rtol=0.0, atol=1e-6):
    raise RuntimeError('Dense vs sparse predictions differ beyond 1e-6 on probe. STOP.')

# Free probe models before the full fit.
del model_dense_probe, model_sparse_probe, pre_dense, pre_sparse, D, S
import gc
gc.collect()

pre = make_preprocessor(sparse_output=True)
model = Pipeline([
    ('preprocess', pre),
    ('regressor', build_model('custom_random_forest', MODEL_PARAMS)),
])
print('Model type:', type(model.named_steps['regressor']).__name__)
print('Model source:', model.named_steps['regressor'].__class__.__module__)
print('Sparse probe passed. Full fit may start; actual peak RAM is not guaranteed.')


## 5. Huấn luyện toàn bộ TRAIN và đánh giá DEV trên VAL

Dùng đúng 302.471 hàng TRAIN, giữ One-Hot không giới hạn category, model/seed và các tham số như bản trước. Không đọc TEST. Nếu model hoặc cây bên trong âm thầm chuyển sparse thành dense khi fit, Kaggle vẫn có thể vượt RAM; kiểm tra tương đương trên mẫu không loại bỏ hoàn toàn rủi ro này.


In [ ]:
start = time.perf_counter()
model.fit(X_train, y_train)  # Fit preprocessing & model ONLY on train data.
fit_seconds = time.perf_counter() - start

trained_pre = model.named_steps['preprocess']
trained_encoder = trained_pre.named_transformers_['cat'].named_steps['encode']
encoded_feature_count = len(numeric) + len(trained_encoder.get_feature_names_out(categorical))
print('Actual FULL one-hot feature count:', encoded_feature_count)
if encoded_feature_count != feature_dim_estimate:
    print('INFO: OHE output count differs from nunique estimate; check imputation/missing categories.')

# Verify the fitted full preprocessor still emits sparse float64 (on a tiny batch).
X_check = trained_pre.transform(X_train.iloc[:32])
if not sparse.issparse(X_check) or X_check.dtype != np.float64:
    raise RuntimeError('Full fitted preprocessor is not sparse float64. STOP.')
print('Final model input matrix type:', type(X_check).__name__, '| dtype:', X_check.dtype)

val_preds = []
for start_index in range(0, len(X_val), VAL_PRED_BATCH):
    batch = X_val.iloc[start_index:start_index + VAL_PRED_BATCH]
    val_preds.append(np.asarray(model.predict(batch)))
pred_val = np.concatenate(val_preds)
metrics = regression_metrics(y_val, pred_val)

print(f'Fit completed in {fit_seconds:,.1f} seconds')
print('DEV validation metrics (NOT FINAL):', metrics)
print('Custom RF OOB:', getattr(model.named_steps['regressor'], 'oob_score_', 'not available'))
print('TEST HAS NOT BEEN USED.')


## 6. Lưu run, dự đoán validation và model vào output

In [ ]:
out_root = (Path('/kaggle/working/mlrf_runs')
            if Path('/kaggle/working').exists()
            else REPO / 'outputs' / 'mlrf_runs')
out_root.mkdir(parents=True, exist_ok=True)

commit = subprocess.run(
    ['git', '-C', str(REPO), 'rev-parse', 'HEAD'],
    capture_output=True, text=True, check=False,
)
git_commit = commit.stdout.strip() if commit.returncode == 0 else None

dataset_identity = {
    'source': 'kaggle:ncd426/machinelearning',
    'path': str(DATA_DIR),
    'approved': False,
    'rows': len(train_use),
    'train_rows_total': len(train_df),
    'train_rows_used': len(train_use),
    'val_rows_used': len(val_use),
    'test_used': False,
    'missing_contract_features': missing_contract_features,
    'contract_match': len(missing_contract_features) == 0,
}

run_config = {
    'model': 'custom_random_forest',
    **MODEL_PARAMS,
    'seed': SEED,
    'split_status': 'pre-split data; validation only, no test used',
    'preprocessing_status': 'provisional adapter: full-category sparse one-hot + float64; NOT TV2 final',
    'schema_status': 'DEV waiver: missing contract features' if missing_contract_features else 'feature names match contract',
    'missing_contract_features': missing_contract_features,
    'training_seconds': fit_seconds,
    'preprocessing_dtype': 'float64',
    'preprocessing_encoding': 'onehot_full_sparse',
    'full_ohe_dimensions_estimated': feature_dim_estimate,
    'full_ohe_dimensions_actual': encoded_feature_count,
    'category_grouping': False,
    'sparse_csr_storage_upper_mib_estimate': float(upper_csr_mib),
    'dense_sparse_probe_prediction_max_abs_diff': max_diff,
    'ram_available_mb_at_preflight': float(available_mb),
    'full_train': len(train_use) == len(train_df),
}

run_path = Path(save_run(
    out_root,
    RUN_ID,
    config=run_config,
    metrics=metrics,
    dataset=dataset_identity,
    git_commit=git_commit,
    predictions={
        'row_id': np.arange(len(val_use), dtype=np.int64),
        'actual': y_val.to_numpy(),
        'predicted': np.asarray(pred_val),
    },
))

model_path = run_path / 'model.joblib'
joblib.dump(model, model_path, compress=3)
print('Artifacts:', run_path)
print('Model:', model_path)
print('Git commit:', git_commit)
print('DEV ONLY: dữ liệu/preprocessing chưa được xác nhận approved.')


## 7. Chạy đúng quy trình local → GitHub → Kaggle

1. Lưu file này thành `notebooks/02_train_real.ipynb` **ở local**, thay bản có `max_categories=256`. Commit và push lên branch `feat/train-real-data` (không push trực tiếp lên `dev`).
2. Trên Kaggle, cập nhật branch và chạy notebook. Nếu probe dense/sparse không qua, notebook sẽ dừng; **không** tự tắt probe và không tự tăng RAM cap để vượt qua lỗi.
3. Kiểm tra `Actual FULL one-hot feature count` và `Final model input matrix type`. Không có category nào bị gộp, nhưng hai feature `characteristics`, `body_type_name` vốn đã không tồn tại trong Parquet vẫn đang được bỏ qua theo DEV waiver.
4. Lưu `model.joblib`, metrics và Git commit. Không dùng TEST trước khi schema và preprocessing được nhóm xác nhận.

**Quan trọng:** Chuyển dense sang sparse giữ các giá trị đầu vào về mặt toán học, nhưng không thể cam kết model huấn luyện cho kết quả đúng từng bit giữa hai cách biểu diễn ở mọi trường hợp. Nếu cần bảo đảm giữ nguyên triển khai dense, hãy train trên máy RAM lớn hơn.
